## Objective 

This notebook performs paired statistical hypothesis testing to determine whether the proposed cross-attention multimodal model significantly outperforms multiple image-only backbone models.

The following baseline models are evaluated:

- CLIP
- ConvNeXt
- Swin
- ViT

For each comparison (cross_attention vs baseline), three paired statistical tests are conducted:

1. Paired t-test (parametric test)
2. Wilcoxon signed-rank test (non-parametric test)
3. Paired bootstrap test (resampling-based test)

All tests are one-sided and evaluate whether the cross-attention model performs better than the baseline model.

## Input Data

Paths are explicitly defined as:

Proposed model:
results/proposed/cross_attention/all_seeds_summary.csv

Baselines:
results/image_only/clip/all_seeds_summary.csv
results/image_only/convnext/all_seeds_summary.csv
results/image_only/swin/all_seeds_summary.csv
results/image_only/vit/all_seeds_summary.csv

## Output

All results will be saved to:

results/statistical_test/

Generated files:
- paired_t_test_results.csv
- wilcoxon_results.csv
- bootstrap_results.csv
- statistical_summary_table.csv

## Hypothesis

For each metric:

Null Hypothesis (H0):
The mean performance difference between cross_attention and baseline is less than or equal to zero.

Alternative Hypothesis (H1):
The cross_attention model performs better than the baseline.

All tests are paired and one-sided.

## Imports and Paths

In [3]:
import os
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon

# Explicit paths
PROPOSED_PATH = "results/proposed/cross_attention/all_seeds_summary.csv"

MODEL_PATHS = {
    "clip": "results/image_only/clip/all_seeds_summary.csv",
    "convnext": "results/image_only/convnext/all_seeds_summary.csv",
    "swin": "results/image_only/swin/all_seeds_summary.csv",
    "vit": "results/image_only/vit/all_seeds_summary.csv"
}

OUTPUT_DIR = "results/statistical_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

METRICS = ["macro_f1", "accuracy"]
ALPHA = 0.05
BOOTSTRAP_ITERATIONS = 10000
RANDOM_SEED = 42

proposed_df = pd.read_csv(PROPOSED_PATH)

## Paired t-test

### Objective

To test whether the mean difference between the cross_attention model and a baseline model is significantly greater than zero.

### What is being tested?

For each seed:

difference = proposed_metric - baseline_metric

We test whether the average of these differences is greater than zero.

### Assumption

The paired differences follow an approximately normal distribution.

### Input

- Per-seed macro_f1 and accuracy
- Matched seeds across models

### Output

- Mean difference
- t-statistic
- p-value (one-sided)
- Decision to reject H0

In [4]:
t_test_results = []

for model_name, model_path in MODEL_PATHS.items():
    baseline_df = pd.read_csv(model_path)
    
    merged = proposed_df.merge(
        baseline_df,
        on="seed",
        suffixes=("_proposed", "_baseline")
    ).sort_values("seed")
    
    for metric in METRICS:
        d = merged[f"{metric}_proposed"] - merged[f"{metric}_baseline"]
        d = d.values
        
        result = ttest_1samp(d, 0.0, alternative="greater")
        
        t_test_results.append({
            "comparison": f"cross_attention vs {model_name}",
            "metric": metric,
            "n": len(d),
            "mean_difference": np.mean(d),
            "t_statistic": result.statistic,
            "p_value": result.pvalue,
            "reject_H0": result.pvalue < ALPHA
        })

t_test_df = pd.DataFrame(t_test_results)
t_test_df.to_csv(os.path.join(OUTPUT_DIR, "paired_t_test_results.csv"), index=False)

t_test_df

,comparison,metric,n,mean_difference,t_statistic,p_value,reject_H0
0,cross_attention vs clip,macro_f1,10,-0.000389,-0.130045,5.503046e-01,False
1,cross_attention vs clip,accuracy,10,-0.000505,-0.165647,5.639521e-01,False
2,cross_attention vs convnext,macro_f1,10,0.067655,23.173583,1.234156e-09,True
3,cross_attention vs convnext,accuracy,10,0.069728,24.002639,9.035813e-10,True
4,cross_attention vs swin,macro_f1,10,0.051924,13.749509,1.198394e-07,True
5,cross_attention vs swin,accuracy,10,0.052624,14.350645,8.279055e-08,True
6,cross_attention vs vit,macro_f1,10,0.247781,15.853746,3.484435e-08,True
7,cross_attention vs vit,accuracy,10,0.248385,15.859928,3.472613e-08,True


## Wilcoxon Signed-Rank Test

### Objective

To test whether the median paired difference is significantly greater than zero without assuming normality.

### Why use this test?

This test does not require the differences to follow a normal distribution.

### Input

- Paired per-seed differences

### Output

- Wilcoxon statistic
- p-value
- Decision regarding H0

In [5]:
wilcoxon_results = []

for model_name, model_path in MODEL_PATHS.items():
    baseline_df = pd.read_csv(model_path)
    
    merged = proposed_df.merge(
        baseline_df,
        on="seed",
        suffixes=("_proposed", "_baseline")
    ).sort_values("seed")
    
    for metric in METRICS:
        d = merged[f"{metric}_proposed"] - merged[f"{metric}_baseline"]
        d = d.values
        
        result = wilcoxon(d, alternative="greater", zero_method="wilcox")
        
        wilcoxon_results.append({
            "comparison": f"cross_attention vs {model_name}",
            "metric": metric,
            "n": len(d),
            "median_difference": np.median(d),
            "wilcoxon_statistic": result.statistic,
            "p_value": result.pvalue,
            "reject_H0": result.pvalue < ALPHA
        })

wilcoxon_df = pd.DataFrame(wilcoxon_results)
wilcoxon_df.to_csv(os.path.join(OUTPUT_DIR, "wilcoxon_results.csv"), index=False)

wilcoxon_df

,comparison,metric,n,median_difference,wilcoxon_statistic,p_value,reject_H0
0,cross_attention vs clip,macro_f1,10,0.001451,28.0,0.500000,False
1,cross_attention vs clip,accuracy,10,0.001514,27.0,0.528320,False
2,cross_attention vs convnext,macro_f1,10,0.069092,55.0,0.000977,True
3,cross_attention vs convnext,accuracy,10,0.072149,55.0,0.000977,True
4,cross_attention vs swin,macro_f1,10,0.053702,55.0,0.000977,True
5,cross_attention vs swin,accuracy,10,0.054490,55.0,0.000977,True
6,cross_attention vs vit,macro_f1,10,0.257414,55.0,0.000977,True
7,cross_attention vs vit,accuracy,10,0.262614,55.0,0.000977,True


## Paired Bootstrap Test

### Objective

To estimate the confidence interval of the mean performance difference using resampling.

### Method

1. Compute paired differences.
2. Sample with replacement.
3. Compute mean difference for each resample.
4. Estimate 95% confidence interval.
5. Compute one-sided bootstrap p-value.

### Input

- Paired differences

### Output

- 95% confidence interval
- Bootstrap p-value
- Decision regarding H0

In [6]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_results = []

for model_name, model_path in MODEL_PATHS.items():
    baseline_df = pd.read_csv(model_path)
    
    merged = proposed_df.merge(
        baseline_df,
        on="seed",
        suffixes=("_proposed", "_baseline")
    ).sort_values("seed")
    
    for metric in METRICS:
        d = merged[f"{metric}_proposed"] - merged[f"{metric}_baseline"]
        d = d.values
        n = len(d)
        
        boot_means = []
        for _ in range(BOOTSTRAP_ITERATIONS):
            sample = rng.choice(d, size=n, replace=True)
            boot_means.append(np.mean(sample))
        
        boot_means = np.array(boot_means)
        
        ci_lower = np.percentile(boot_means, 2.5)
        ci_upper = np.percentile(boot_means, 97.5)
        p_value = (1 + np.sum(boot_means <= 0)) / (BOOTSTRAP_ITERATIONS + 1)
        
        bootstrap_results.append({
            "comparison": f"cross_attention vs {model_name}",
            "metric": metric,
            "mean_difference": np.mean(d),
            "ci_95_lower": ci_lower,
            "ci_95_upper": ci_upper,
            "bootstrap_p_value": p_value,
            "reject_H0": p_value < ALPHA
        })

bootstrap_df = pd.DataFrame(bootstrap_results)
bootstrap_df.to_csv(os.path.join(OUTPUT_DIR, "bootstrap_results.csv"), index=False)

bootstrap_df

,comparison,metric,mean_difference,ci_95_lower,ci_95_upper,bootstrap_p_value,reject_H0
0,cross_attention vs clip,macro_f1,-0.000389,-0.005948,0.004894,0.554545,False
1,cross_attention vs clip,accuracy,-0.000505,-0.006408,0.004995,0.562044,False
2,cross_attention vs convnext,macro_f1,0.067655,0.062151,0.072864,0.000100,True
3,cross_attention vs convnext,accuracy,0.069728,0.064279,0.074825,0.000100,True
4,cross_attention vs swin,macro_f1,0.051924,0.044717,0.058750,0.000100,True
5,cross_attention vs swin,accuracy,0.052624,0.045510,0.059132,0.000100,True
6,cross_attention vs vit,macro_f1,0.247781,0.217866,0.275197,0.000100,True
7,cross_attention vs vit,accuracy,0.248385,0.217356,0.275632,0.000100,True


## Summary Table

In [7]:
summary = t_test_df.merge(
    wilcoxon_df[["comparison", "metric", "p_value"]],
    on=["comparison", "metric"],
    suffixes=("_t_test", "_wilcoxon")
)

summary = summary.merge(
    bootstrap_df[["comparison", "metric", "bootstrap_p_value", "ci_95_lower", "ci_95_upper"]],
    on=["comparison", "metric"]
)

summary.to_csv(os.path.join(OUTPUT_DIR, "statistical_summary_table.csv"), index=False)

summary

,comparison,metric,n,mean_difference,t_statistic,p_value_t_test,reject_H0,p_value_wilcoxon,bootstrap_p_value,ci_95_lower,ci_95_upper
0,cross_attention vs clip,macro_f1,10,-0.000389,-0.130045,5.503046e-01,False,0.500000,0.554545,-0.005948,0.004894
1,cross_attention vs clip,accuracy,10,-0.000505,-0.165647,5.639521e-01,False,0.528320,0.562044,-0.006408,0.004995
2,cross_attention vs convnext,macro_f1,10,0.067655,23.173583,1.234156e-09,True,0.000977,0.000100,0.062151,0.072864
3,cross_attention vs convnext,accuracy,10,0.069728,24.002639,9.035813e-10,True,0.000977,0.000100,0.064279,0.074825
4,cross_attention vs swin,macro_f1,10,0.051924,13.749509,1.198394e-07,True,0.000977,0.000100,0.044717,0.058750
5,cross_attention vs swin,accuracy,10,0.052624,14.350645,8.279055e-08,True,0.000977,0.000100,0.045510,0.059132
6,cross_attention vs vit,macro_f1,10,0.247781,15.853746,3.484435e-08,True,0.000977,0.000100,0.217866,0.275197
7,cross_attention vs vit,accuracy,10,0.248385,15.859928,3.472613e-08,True,0.000977,0.000100,0.217356,0.275632
